# Fundamentos de Escalabilidad y Rendimiento

## Objetivos de Aprendizaje

- Comprender los conceptos de **escalabilidad** (vertical y horizontal)
- Conocer los **3 V del Big Data** (Volumen, Velocidad, Variedad)
- Identificar métricas básicas de rendimiento
- Introducción a conceptos de **costo** en arquitecturas de datos
- Entender trade-offs entre rendimiento y costo

## 1. ¿Qué es la Escalabilidad?

**Escalabilidad** es la capacidad de un sistema para manejar crecimiento de datos, usuarios o transacciones sin degradación significativa del rendimiento.

### Tipos de Escalabilidad

| Tipo | Descripción | Ejemplo | Ventajas | Desventajas |
|------|-------------|---------|----------|-------------|
| **Vertical (Scale Up)** | Aumentar capacidad del servidor (CPU, RAM, disco) | Pasar de 16GB RAM a 64GB RAM | Simple de implementar, sin cambios arquitectónicos | Límite físico de hardware, punto único de fallo |
| **Horizontal (Scale Out)** | Agregar más servidores/nodos al sistema | Pasar de 3 nodos a 10 nodos | Sin límite teórico, alta disponibilidad | Mayor complejidad, necesita distribución de datos |

## 2. Los 3 V del Big Data

Los **3 V** son dimensiones fundamentales que definen los requisitos de escalabilidad:

### 📊 **Volumen** (Volume)
- **Definición:** Cantidad total de datos almacenados
- **Ejemplo:** 1 TB vs 100 PB de datos históricos
- **Impacto:** Requiere capacidad de almacenamiento y estrategias de compresión/particionamiento
- **Decisiones:** ¿Cuánto almacenamiento necesito? ¿Cuánto tiempo retener datos?

### ⚡ **Velocidad** (Velocity)
- **Definición:** Tasa de ingesta y procesamiento de datos
- **Ejemplo:** 100 registros/segundo vs 1 millón eventos/segundo
- **Impacto:** Requiere throughput alto y baja latencia
- **Decisiones:** ¿Batch o streaming? ¿Qué latencia tolera el negocio?

### 🎨 **Variedad** (Variety)
- **Definición:** Diversidad de formatos y fuentes de datos
- **Ejemplo:** Datos estructurados (SQL), semi-estructurados (JSON), no estructurados (logs, imágenes)
- **Impacto:** Requiere flexibilidad en esquemas y procesamiento heterogéneo
- **Decisiones:** ¿Data Warehouse rígido o Data Lake flexible?

---

**Nota:** Algunos autores agregan más V's (Veracidad, Valor, Variabilidad), pero estas 3 son fundamentales para dimensionamiento.

## 3. Dimensionamiento Básico: Recursos Clave

Para escalar una arquitectura de datos, debemos dimensionar tres tipos de recursos:

### 💾 **Almacenamiento (Storage)**
- **Qué dimensionar:** Capacidad en GB/TB/PB, IOPS (operaciones entrada/salida por segundo)
- **Factores:** Volumen de datos, factor de retención (días/años), factor de compresión (típicamente 3-5x)
- **Ejemplo:** 10 TB/mes de logs × 24 meses retención × 0.3 compresión = **72 TB** necesarios

### 🖥️ **Cómputo (Compute)**
- **Qué dimensionar:** CPUs/vCPUs, RAM, aceleración (GPUs para ML)
- **Factores:** Volumen procesado por ventana, complejidad de transformaciones, concurrencia de queries
- **Ejemplo:** Procesar 1 TB/hora requiere clusters grandes; procesar 10 GB/hora puede usar instancias pequeñas

### 🌐 **Red (Network)**
- **Qué dimensionar:** Ancho de banda (Mbps/Gbps), latencia entre regiones
- **Factores:** Velocidad de ingesta, transferencias entre zonas/regiones, replicación
- **Ejemplo:** Ingerir 1 GB/s requiere mínimo 8 Gbps de ancho de banda (más overhead)

In [ ]:
# Ejemplo: Calculadora de dimensionamiento básico
import pandas as pd

# Parámetros del sistema
volumen_diario_gb = 500  # GB ingestados por día
dias_retencion = 365 * 2  # 2 años
factor_compresion = 0.25  # Parquet comprime ~75%
factor_replicas = 3  # HDFS/S3 típicamente 3x

# Cálculo de storage necesario
volumen_total_gb = volumen_diario_gb * dias_retencion
volumen_comprimido_gb = volumen_total_gb * factor_compresion
volumen_con_replicas_gb = volumen_comprimido_gb * factor_replicas

print("=== DIMENSIONAMIENTO DE STORAGE ===")
print(f"Volumen diario: {volumen_diario_gb} GB")
print(f"Retención: {dias_retencion} días")
print(f"Volumen total (sin compresión): {volumen_total_gb:,.0f} GB = {volumen_total_gb/1024:.1f} TB")
print(f"Volumen comprimido (factor {factor_compresion}): {volumen_comprimido_gb:,.0f} GB = {volumen_comprimido_gb/1024:.1f} TB")
print(f"Volumen con réplicas ({factor_replicas}x): {volumen_con_replicas_gb:,.0f} GB = {volumen_con_replicas_gb/1024:.1f} TB")

# Velocidad de ingesta
registros_por_segundo = 5000
bytes_por_registro = 1024  # 1 KB por registro
mbps_necesario = (registros_por_segundo * bytes_por_registro * 8) / (1024 * 1024)

print("\n=== DIMENSIONAMIENTO DE RED ===")
print(f"Registros/segundo: {registros_por_segundo:,}")
print(f"Bytes por registro: {bytes_por_registro}")
print(f"Ancho de banda necesario: {mbps_necesario:.2f} Mbps")
print(f"Recomendación: Provisionar {mbps_necesario * 1.5:.0f} Mbps (50% overhead)")

## 4. Métricas Básicas de Rendimiento

Para medir si un sistema escala correctamente, monitoreamos estas métricas:

| Métrica | Definición | Ejemplo | Objetivo Típico |
|---------|------------|---------|------------------|
| **Throughput** | Volumen procesado por unidad de tiempo | 100 GB/hora | Maximizar |
| **Latencia** | Tiempo desde ingesta hasta disponibilidad | 5 minutos | Minimizar |
| **Tiempo de Query (P95)** | Percentil 95 de tiempo de ejecución de consultas | 2 segundos | < 3s para BI |
| **Disponibilidad (Uptime)** | % de tiempo que el sistema está operativo | 99.9% | ≥ 99.5% |
| **Error Rate** | % de operaciones fallidas | 0.1% | < 1% |

**P95 (Percentil 95):** El 95% de las queries son más rápidas que este valor. Mejor que el promedio porque ignora outliers extremos.

In [ ]:
# Ejemplo: Calcular métricas de rendimiento
import numpy as np
import matplotlib.pyplot as plt

# Simular tiempos de query (en segundos)
np.random.seed(42)
tiempos_query = np.concatenate([
    np.random.gamma(2, 0.5, 950),  # 95% de queries rápidas
    np.random.gamma(5, 2, 50)      # 5% de queries lentas (outliers)
])

# Calcular estadísticas
promedio = np.mean(tiempos_query)
mediana = np.median(tiempos_query)
p95 = np.percentile(tiempos_query, 95)
p99 = np.percentile(tiempos_query, 99)

print("=== MÉTRICAS DE RENDIMIENTO ===")
print(f"Promedio: {promedio:.2f} segundos")
print(f"Mediana (P50): {mediana:.2f} segundos")
print(f"P95: {p95:.2f} segundos")
print(f"P99: {p99:.2f} segundos")
print(f"\n✅ SLO: P95 < 3s → {'CUMPLE' if p95 < 3 else 'NO CUMPLE'}")

# Gráfico de distribución
plt.figure(figsize=(10, 4))
plt.hist(tiempos_query, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(promedio, color='blue', linestyle='--', label=f'Promedio: {promedio:.2f}s')
plt.axvline(p95, color='red', linestyle='--', label=f'P95: {p95:.2f}s')
plt.xlabel('Tiempo de Query (segundos)')
plt.ylabel('Frecuencia')
plt.title('Distribución de Tiempos de Query')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Nota: P95 es más útil que promedio porque ignora outliers extremos.")

## 5. Introducción a Costos en Arquitecturas de Datos

Escalar tiene un costo económico. Debemos entender los **componentes de costo**:

### 💰 Principales Componentes de Costo

1. **Almacenamiento (Storage)**
   - Costo por GB/mes (varía según tipo: Standard, Nearline, Archive)
   - Ejemplo: AWS S3 Standard ~$0.023/GB/mes, Glacier ~$0.004/GB/mes

2. **Cómputo (Compute)**
   - Costo por hora de CPU/RAM (instancias o clusters)
   - Ejemplo: Snowflake ~$2-4/crédito/hora, BigQuery $5/TB procesado

3. **Transferencia de Datos (Egress)**
   - Costo por GB transferido fuera de la nube o entre regiones
   - Ejemplo: AWS $0.09/GB egress, gratis dentro de la misma región

4. **Licencias y Servicios Gestionados**
   - Herramientas de catálogo (Collibra, Alation), BI (Tableau, Looker)
   - Pueden ser $$/usuario/mes o $$/GB procesado

### 🎯 Trade-offs: Rendimiento vs Costo

| Decisión | Impacto en Rendimiento | Impacto en Costo |
|----------|------------------------|------------------|
| Aumentar tamaño de cluster | ⬆️ Queries más rápidas | ⬆️ Mayor costo/hora |
| Comprimir datos (Parquet) | ➡️ Slight overhead en escritura, más rápido en lectura | ⬇️ Menos storage, menos bytes procesados |
| Particionar tablas por fecha | ⬆️ Queries filtradas son mucho más rápidas | ➡️ Mismo costo storage, menor costo compute |
| Usar storage Archive (fríos) | ⬇️ Acceso muy lento (horas) | ⬇️ 80% más barato que Standard |
| Auto-suspend clusters | ➡️ Cold start al despertar (segundos) | ⬇️ No pagar cuando no se usa |

In [ ]:
# Ejemplo: Simulación simple de costo mensual
# Supongamos un Data Warehouse en la nube

# Parámetros
storage_tb = 50  # TB almacenados
costo_storage_gb_mes = 0.023  # USD por GB/mes (S3 Standard)

compute_horas_mes = 200  # horas de cluster activo
costo_compute_hora = 3.5  # USD por hora (cluster mediano)

transferencia_gb_mes = 1000  # GB de egress (reportes descargados)
costo_egress_gb = 0.09  # USD por GB

# Cálculos
costo_storage = storage_tb * 1024 * costo_storage_gb_mes
costo_compute = compute_horas_mes * costo_compute_hora
costo_egress = transferencia_gb_mes * costo_egress_gb
costo_total = costo_storage + costo_compute + costo_egress

print("=== ESTIMACIÓN DE COSTO MENSUAL ===")
print(f"Storage: {storage_tb} TB × ${costo_storage_gb_mes}/GB/mes = ${costo_storage:,.2f}")
print(f"Compute: {compute_horas_mes} horas × ${costo_compute_hora}/hora = ${costo_compute:,.2f}")
print(f"Egress: {transferencia_gb_mes} GB × ${costo_egress_gb}/GB = ${costo_egress:,.2f}")
print(f"\n💵 TOTAL MENSUAL: ${costo_total:,.2f}")

# Desglose porcentual
print("\n=== DESGLOSE PORCENTUAL ===")
print(f"Storage: {costo_storage/costo_total*100:.1f}%")
print(f"Compute: {costo_compute/costo_total*100:.1f}%")
print(f"Egress: {costo_egress/costo_total*100:.1f}%")

# Gráfico de desglose
costos = [costo_storage, costo_compute, costo_egress]
labels = ['Storage', 'Compute', 'Egress']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

plt.figure(figsize=(8, 6))
plt.pie(costos, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors)
plt.title(f'Desglose de Costo Mensual: ${costo_total:,.0f}')
plt.tight_layout()
plt.show()

print("\n💡 Optimización: Compute es el mayor costo. Considera auto-suspend y right-sizing.")

## 6. Estrategias Básicas de Optimización

### 🚀 Para Mejorar Rendimiento
1. **Particionar datos** por fecha o categoría (reduce escaneo)
2. **Usar formatos columnares** (Parquet, ORC) en lugar de CSV
3. **Indexar o clusterizar** columnas frecuentemente filtradas
4. **Proyectar solo columnas necesarias** (evitar `SELECT *`)
5. **Cachear resultados** de queries repetitivas

### 💰 Para Reducir Costos
1. **Auto-suspend/resume** de clusters cuando no se usan
2. **Right-sizing:** Usar tamaño de cluster adecuado (no sobre-provisionar)
3. **Storage tiering:** Mover datos fríos a Archive (S3 Glacier, Azure Cool)
4. **Comprimir datos:** Parquet con Snappy/Gzip (3-5x reducción)
5. **Limitar egress:** Procesar datos dentro de la misma región

### ⚖️ Balance Rendimiento-Costo
- **No siempre más rápido es mejor:** Un sistema 10% más rápido pero 50% más caro no es óptimo
- **Definir SLOs realistas:** Si el negocio tolera 5 minutos de latencia, no pagar por 30 segundos
- **Medir y iterar:** Monitorear métricas de rendimiento y costo continuamente

## 7. Ejercicio Práctico

**Escenario:** Tu empresa procesa logs de aplicación web:
- **Volumen:** 200 GB/día de logs JSON
- **Velocidad:** 10,000 eventos/segundo en picos
- **Variedad:** Logs estructurados + algunos campos JSON anidados
- **Retención:** 90 días en hot storage, 2 años en cold storage
- **SLO:** Queries de dashboard < 5 segundos (P95)

**Tareas:**
1. Calcular storage necesario (hot + cold) considerando compresión 4x
2. Determinar ancho de banda de red necesario para picos
3. Proponer estrategia de particionamiento
4. Identificar 3 optimizaciones de costo sin afectar SLO
5. Listar 3 métricas clave para monitorear

In [ ]:
# Plantilla de solución
plantilla = '''
=== SOLUCIÓN DEL EJERCICIO ===

1. DIMENSIONAMIENTO DE STORAGE:
   Hot (90 días): [calcular GB/TB]
   Cold (2 años menos 90 días): [calcular GB/TB]
   Total considerando compresión 4x: [resultado]

2. ANCHO DE BANDA (PICOS):
   Eventos/segundo: 10,000
   Tamaño promedio evento: [estimar KB]
   Mbps necesario: [calcular]
   Recomendación con overhead: [valor]

3. ESTRATEGIA DE PARTICIONAMIENTO:
   [Proponer: por fecha diaria/horaria, por tipo de log, etc.]

4. OPTIMIZACIONES DE COSTO:
   a) [optimización 1]
   b) [optimización 2]
   c) [optimización 3]

5. MÉTRICAS CLAVE:
   a) [métrica 1]
   b) [métrica 2]
   c) [métrica 3]
'''

print(plantilla)

# TIP: Usa las fórmulas de los ejemplos anteriores como referencia

## 8. Resumen y Próximos Pasos

### ✅ Conceptos Clave Aprendidos
- **Escalabilidad vertical vs horizontal:** Cuándo usar cada una
- **3 V del Big Data:** Volumen, Velocidad, Variedad como ejes de dimensionamiento
- **Recursos a dimensionar:** Storage, Compute, Network
- **Métricas de rendimiento:** Throughput, latencia, P95, disponibilidad
- **Componentes de costo:** Storage, compute, egress y trade-offs

### 🎯 Próximos Temas (Nivel Mid)
- Cálculos detallados de dimensionamiento con casos reales
- Técnicas de optimización (particionamiento, clustering, compresión)
- Introducción a FinOps: presupuestos, alertas, chargeback/showback
- Observabilidad: dashboards de métricas y alertas automáticas

### 📚 Recursos Recomendados
- AWS Well-Architected Framework (Performance Efficiency pillar)
- Google Cloud Architecture Center (Cost Optimization best practices)
- "Designing Data-Intensive Applications" - Martin Kleppmann (Capítulo 1)
- Calculadoras de costo: AWS Pricing Calculator, Azure Pricing Calculator, GCP Pricing Calculator